# NYC Taxi Data Analysis - dlt Workshop Homework

This notebook analyzes NYC Yellow Taxi trip data loaded from a custom API using dlt (data load tool).

**Homework Questions:**
1. What is the start date and end date of the dataset?
2. What proportion of trips are paid with credit card?
3. What is the total amount of money generated in tips?

**Data Source:** Custom API endpoint (`https://us-central1-dlthub-analytics.cloudfunctions.net/data_engineering_zoomcamp_api`)

## 1. Import Required Libraries

Import DuckDB for querying the data and pandas for data analysis.

In [5]:
import duckdb

# Connect to the DuckDB database created by dlt pipeline
conn = duckdb.connect('taxi_pipeline.duckdb')

print("✅ Connected to DuckDB database successfully!")

✅ Connected to DuckDB database successfully!


## 2. Explore the Data

Let's first understand the structure of our data and see what tables and columns we have.

In [6]:
# Check available tables
tables = conn.execute("SHOW TABLES").fetchdf()
print("Available tables:")
print(tables)
print()

# Get table schema
schema = conn.execute("DESCRIBE nyc_taxi_data.trips").fetchdf()
print("Trips table schema:")
print(schema)
print()

# Get row count
row_count = conn.execute("SELECT COUNT(*) as count FROM nyc_taxi_data.trips").fetchone()[0]
print(f"Total records: {row_count:,}")
print()

# Show sample data
sample = conn.execute("SELECT * FROM nyc_taxi_data.trips LIMIT 5").fetchdf()
print("Sample data:")
sample

Available tables:
Empty DataFrame
Columns: [name]
Index: []

Trips table schema:
               column_name               column_type null   key default extra
0                  end_lat                    DOUBLE  YES  None    None  None
1                  end_lon                    DOUBLE  YES  None    None  None
2                 fare_amt                    DOUBLE  YES  None    None  None
3          passenger_count                    BIGINT  YES  None    None  None
4             payment_type                   VARCHAR  YES  None    None  None
5                start_lat                    DOUBLE  YES  None    None  None
6                start_lon                    DOUBLE  YES  None    None  None
7                  tip_amt                    DOUBLE  YES  None    None  None
8                tolls_amt                    DOUBLE  YES  None    None  None
9                total_amt                    DOUBLE  YES  None    None  None
10           trip_distance                    DOUBLE  YES  No

,end_lat,end_lon,fare_amt,passenger_count,payment_type,start_lat,start_lon,tip_amt,tolls_amt,total_amt,trip_distance,trip_dropoff_date_time,trip_pickup_date_time,surcharge,vendor_name,_dlt_load_id,_dlt_id,store_and_forward
0,40.742963,-73.980072,45.0,1,Credit,40.641525,-73.787442,9.0,4.15,58.15,17.52,2009-06-15 01:48:00+02:00,2009-06-15 01:23:00+02:00,0.0,VTS,1771882758.985973,iUam2mJr9s/dXQ,NaN
1,40.740187,-74.005698,6.5,1,Credit,40.722065,-74.009767,1.0,0.00,8.50,1.56,2009-06-18 19:43:00+02:00,2009-06-18 19:35:00+02:00,1.0,VTS,1771882758.985973,cRulNgenqxa2FQ,NaN
2,40.718043,-74.004745,12.5,5,Credit,40.761945,-73.983038,2.0,0.00,15.50,3.37,2009-06-10 20:27:00+02:00,2009-06-10 20:08:00+02:00,1.0,VTS,1771882758.985973,ZY/knbsiVCTbig,NaN
3,40.739637,-73.985233,4.9,1,CASH,40.749802,-73.992247,0.0,0.00,5.40,1.11,2009-06-15 01:58:00+02:00,2009-06-15 01:54:00+02:00,0.5,VTS,1771882758.985973,+2K8mDe3CMAqGg,NaN
4,40.730032,-73.852693,25.7,1,CASH,40.776825,-73.949233,0.0,4.15,29.85,11.09,2009-06-13 15:23:00+02:00,2009-06-13 15:01:00+02:00,0.0,VTS,1771882758.985973,2WXZHlTAqpxdUQ,NaN


## Question 1: What is the start date and end date of the dataset?

Find the minimum and maximum dates in the `trip_pickup_date_time` column.

In [7]:
# Query to find date range (FIXED: column name is trip_pickup_date_time)
query = """
SELECT 
    DATE(MIN(trip_pickup_date_time)) as start_date,
    DATE(MAX(trip_pickup_date_time)) as end_date
FROM nyc_taxi_data.trips
"""

result = conn.execute(query).fetchdf()
start_date = str(result['start_date'][0])
end_date = str(result['end_date'][0])

print("="*60)
print("QUESTION 1: Dataset Date Range")
print("="*60)
print(f"Start Date: {start_date}")
print(f"End Date:   {end_date}")
print("="*60)
print()

# Determine which option is correct
if "2009-01-01" in start_date and "2009-01-31" in end_date:
    print("✅ Answer: 2009-01-01 to 2009-01-31")
elif "2009-06-01" in start_date and ("2009-07-01" in end_date or "2009-06" in end_date):
    print("✅ Answer: 2009-06-01 to 2009-07-01")
elif "2024-01-01" in start_date and "2024-02-01" in end_date:
    print("✅ Answer: 2024-01-01 to 2024-02-01")
elif "2024-06-01" in start_date and "2024-07-01" in end_date:
    print("✅ Answer: 2024-06-01 to 2024-07-01")

QUESTION 1: Dataset Date Range
Start Date: 2009-06-01 00:00:00
End Date:   2009-07-01 00:00:00

✅ Answer: 2009-06-01 to 2009-07-01


## Question 2: What proportion of trips are paid with credit card?

Calculate the percentage of trips where `payment_type` is 'Credit' or similar.

In [8]:
# First, let's see what payment types we have
payment_types = conn.execute("""
SELECT payment_type, COUNT(*) as count
FROM nyc_taxi_data.trips
GROUP BY payment_type
ORDER BY count DESC
""").fetchdf()

print("Payment types in the dataset:")
print(payment_types)
print()

# Calculate credit card proportion
query = """
SELECT 
    COUNT(*) as total_trips,
    SUM(CASE WHEN LOWER(payment_type) LIKE '%credit%' OR LOWER(payment_type) LIKE '%crd%' THEN 1 ELSE 0 END) as credit_trips,
    ROUND(100.0 * SUM(CASE WHEN LOWER(payment_type) LIKE '%credit%' OR LOWER(payment_type) LIKE '%crd%' THEN 1 ELSE 0 END) / COUNT(*), 2) as percentage
FROM nyc_taxi_data.trips
"""

result = conn.execute(query).fetchdf()
total_trips = result['total_trips'][0]
credit_trips = result['credit_trips'][0]
percentage = result['percentage'][0]

print("="*60)
print("QUESTION 2: Credit Card Payment Proportion")
print("="*60)
print(f"Total trips: {total_trips:,}")
print(f"Credit card trips: {credit_trips:,}")
print(f"Percentage: {percentage}%")
print("="*60)
print()

# Determine which option is correct
if 16.0 <= percentage < 17.0:
    print("✅ Answer: 16.66%")
elif 26.0 <= percentage < 27.0:
    print("✅ Answer: 26.66%")
elif 36.0 <= percentage < 37.0:
    print("✅ Answer: 36.66%")
elif 46.0 <= percentage < 47.0:
    print("✅ Answer: 46.66%")

Payment types in the dataset:
  payment_type  count
0         CASH   7235
1       Credit   2666
2         Cash     97
3      Dispute      1
4    No Charge      1

QUESTION 2: Credit Card Payment Proportion
Total trips: 10,000
Credit card trips: 2,666.0
Percentage: 26.66%

✅ Answer: 26.66%


## Question 3: What is the total amount of money generated in tips?

Sum the `tip_amt` column to find the total tips.

In [9]:
# Query to calculate total tips (FIXED: column name is tip_amt)
query = """
SELECT 
    SUM(tip_amt) as total_tips,
    COUNT(*) as total_trips,
    AVG(tip_amt) as avg_tip
FROM nyc_taxi_data.trips
"""

result = conn.execute(query).fetchdf()
total_tips = result['total_tips'][0]
total_trips = result['total_trips'][0]
avg_tip = result['avg_tip'][0]

print("="*60)
print("QUESTION 3: Total Tips Amount")
print("="*60)
print(f"Total tips: ${total_tips:,.2f}")
print(f"Average tip per trip: ${avg_tip:,.2f}")
print(f"Total trips: {total_trips:,}")
print("="*60)
print()

# Determine which option is correct
if 4000 <= total_tips < 5000:
    print("✅ Answer: $4,063.41")
elif 6000 <= total_tips < 7000:
    print("✅ Answer: $6,063.41")
elif 8000 <= total_tips < 9000:
    print("✅ Answer: $8,063.41")
elif 10000 <= total_tips < 11000:
    print("✅ Answer: $10,063.41")

QUESTION 3: Total Tips Amount
Total tips: $6,063.41
Average tip per trip: $0.61
Total trips: 10,000

✅ Answer: $6,063.41


## Summary

Homework answers for dlt Workshop - Data Engineering Zoomcamp 2026

**Pipeline:** Successfully loaded 10,000 NYC taxi trip records from custom API into DuckDB using dlt

In [10]:
# Close the database connection
conn.close()
print("✅ Database connection closed")

✅ Database connection closed
